# Knapsack: Greedy Strategies Compared

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Colors ─────────────────────────────────────────────────────────────
C_AVAILABLE  = '#bdbdbd'   # gray  – still available
C_SELECTED   = '#4caf50'   # green – already selected
C_NO_FIT     = '#e53935'   # red   – doesn't fit
C_CURRENT    = '#1e88e5'   # blue  – being considered now
C_PARTIAL    = '#90caf9'   # light blue – fractional portion
C_STRATEGIES = ['#1e88e5', '#e53935', '#4caf50', '#ff9800']  # bars in summary

# ── Matplotlib defaults ───────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'monospace',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
})

print('Setup OK')

In [ ]:
# ── Greedy solvers ────────────────────────────────────────────────────

def greedy_steps(weights, values, W, key_fn):
    """
    Run a 0/1 greedy strategy.  Returns a list of step dicts:
      { 'order': [...], 'selected': set, 'rejected': set,
        'current': int|None, 'rem_cap': int, 'acc_val': int, 'acc_wt': int }
    The first entry is the initial state; the last is the final result.
    """
    n = len(weights)
    order = sorted(range(n), key=key_fn, reverse=True)
    selected, rejected = set(), set()
    rem, val, wt = W, 0, 0
    steps = [dict(order=order, selected=set(selected), rejected=set(rejected),
                  current=None, rem_cap=rem, acc_val=val, acc_wt=wt)]
    for idx in order:
        if weights[idx] <= rem:
            selected.add(idx)
            rem -= weights[idx]
            val += values[idx]
            wt  += weights[idx]
        else:
            rejected.add(idx)
        steps.append(dict(order=order, selected=set(selected), rejected=set(rejected),
                          current=idx, rem_cap=rem, acc_val=val, acc_wt=wt))
    return steps


def fractional_steps(weights, values, W):
    """
    Fractional knapsack (greedy by ratio, allows partial items).
    Returns steps like above but with extra 'fractions' dict {idx: frac}.
    """
    n = len(weights)
    order = sorted(range(n), key=lambda i: values[i]/weights[i], reverse=True)
    fractions = {}  # idx -> fraction taken (0..1)
    selected, rejected = set(), set()
    rem, val, wt = float(W), 0.0, 0.0
    steps = [dict(order=order, selected=set(), rejected=set(),
                  current=None, rem_cap=rem, acc_val=val, acc_wt=wt,
                  fractions=dict(fractions))]
    for idx in order:
        if weights[idx] <= rem:
            selected.add(idx)
            fractions[idx] = 1.0
            rem -= weights[idx]
            val += values[idx]
            wt  += weights[idx]
        elif rem > 0:
            frac = rem / weights[idx]
            selected.add(idx)
            fractions[idx] = frac
            val += values[idx] * frac
            wt  += weights[idx] * frac
            rem = 0
        else:
            rejected.add(idx)
            fractions[idx] = 0.0
        steps.append(dict(order=order, selected=set(selected), rejected=set(rejected),
                          current=idx, rem_cap=rem, acc_val=val, acc_wt=wt,
                          fractions=dict(fractions)))
    return steps


def solve_all(weights, values, W):
    """Return {name: steps} for all 4 strategies."""
    return {
        'Max Value':          greedy_steps(weights, values, W,
                                          key_fn=lambda i: values[i]),
        'Min Weight':         greedy_steps(weights, values, W,
                                          key_fn=lambda i: -weights[i]),
        'Best Ratio':         greedy_steps(weights, values, W,
                                          key_fn=lambda i: values[i]/weights[i]),
        'Fractional (exact)': fractional_steps(weights, values, W),
    }

print('Solvers OK')

In [ ]:
# ── Visualization helpers ─────────────────────────────────────────────

def draw_step(ax, weights, values, W, step, is_fractional=False):
    """Draw item bars for one greedy step."""
    n = len(weights)
    ax.clear()
    labels = [f'Item {i}\nw={weights[i]} v={values[i]}\nr={values[i]/weights[i]:.2f}'
              for i in range(n)]
    bar_vals = list(values)

    colors = []
    for i in range(n):
        if i == step.get('current'):
            colors.append(C_CURRENT)
        elif i in step['selected']:
            colors.append(C_SELECTED)
        elif i in step['rejected']:
            colors.append(C_NO_FIT)
        else:
            colors.append(C_AVAILABLE)

    bars = ax.bar(range(n), bar_vals, color=colors, edgecolor='white', linewidth=1.2)

    # Fractional: overlay a lighter bar for partial items
    if is_fractional and 'fractions' in step:
        for i, frac in step['fractions'].items():
            if 0 < frac < 1:
                full_h = values[i]
                taken_h = values[i] * frac
                # Redraw: bottom part green (taken), top part light
                ax.bar(i, taken_h, color=C_SELECTED, edgecolor='white', linewidth=1.2)
                ax.bar(i, full_h - taken_h, bottom=taken_h, color=C_PARTIAL,
                       edgecolor='white', linewidth=1.2, hatch='//')
                ax.text(i, taken_h + 0.3, f'{frac:.0%}', ha='center',
                        fontsize=9, fontweight='bold')

    ax.set_xticks(range(n))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Value')
    ax.set_title(
        f"Remaining capacity: {step['rem_cap']:.1f}  |  "
        f"Accumulated weight: {step['acc_wt']:.1f}  |  "
        f"Accumulated value: {step['acc_val']:.2f}",
        fontsize=10, loc='left'
    )

    # Legend
    patches = [
        mpatches.Patch(color=C_AVAILABLE, label='Available'),
        mpatches.Patch(color=C_SELECTED,  label='Selected'),
        mpatches.Patch(color=C_NO_FIT,    label="Doesn't fit"),
        mpatches.Patch(color=C_CURRENT,   label='Current pick'),
    ]
    if is_fractional:
        patches.append(mpatches.Patch(color=C_PARTIAL, label='Partial (not taken)'))
    ax.legend(handles=patches, fontsize=8, loc='upper right', framealpha=0.9)


def draw_summary(ax, all_steps, optimum=None):
    """Summary bar chart comparing final values of all strategies."""
    ax.clear()
    names = list(all_steps.keys())
    finals = [all_steps[n][-1]['acc_val'] for n in names]
    bars = ax.bar(range(len(names)), finals, color=C_STRATEGIES[:len(names)],
                  edgecolor='white', linewidth=1.2)
    for b, v in zip(bars, finals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.5, f'{v:.2f}',
                ha='center', fontsize=9, fontweight='bold')
    if optimum is not None:
        ax.axhline(optimum, color='black', linestyle='--', linewidth=1.5, label=f'Optimum = {optimum}')
        ax.legend(fontsize=9)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=9)
    ax.set_ylabel('Total Value')
    ax.set_title('Strategy Comparison', fontsize=12, fontweight='bold')

print('Visualization OK')

In [ ]:
# ── Interactive widget ────────────────────────────────────────────────

# Default instance from the notes
DEFAULT_W  = '100,50,45,20,10,5'
DEFAULT_V  = '40,35,18,4,10,2'
DEFAULT_C  = 100
DEFAULT_OPT = 55

strategy_dd = widgets.Dropdown(
    options=['Max Value', 'Min Weight', 'Best Ratio', 'Fractional (exact)'],
    value='Max Value',
    description='Strategy:',
    style={'description_width': 'initial'},
)

step_slider = widgets.IntSlider(
    value=0, min=0, max=6, step=1,
    description='Step:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)

show_summary_btn = widgets.ToggleButton(
    value=False, description='Show Summary',
    button_style='info', icon='bar-chart',
    layout=widgets.Layout(width='160px'),
)

weights_txt = widgets.Text(
    value=DEFAULT_W, description='Weights:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px'),
)
values_txt = widgets.Text(
    value=DEFAULT_V, description='Values:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px'),
)
cap_txt = widgets.IntText(
    value=DEFAULT_C, description='Capacity W:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px'),
)
opt_txt = widgets.Text(
    value=str(DEFAULT_OPT), description='Optimum:',
    placeholder='blank = none',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px'),
)

out = widgets.Output()

# ── State cache ────────────────────────────────────────────────────────
_cache = {}

def _parse_instance():
    w = [int(x.strip()) for x in weights_txt.value.split(',') if x.strip()]
    v = [int(x.strip()) for x in values_txt.value.split(',') if x.strip()]
    assert len(w) == len(v), 'weights and values must have same length'
    assert all(x > 0 for x in w), 'weights must be positive'
    W = int(cap_txt.value)
    opt_str = opt_txt.value.strip()
    optimum = float(opt_str) if opt_str else None
    return w, v, W, optimum

def _recompute():
    try:
        w, v, W, optimum = _parse_instance()
    except Exception as e:
        with out:
            clear_output(wait=True)
            print(f'Input error: {e}')
        return
    _cache['weights'] = w
    _cache['values']  = v
    _cache['W']       = W
    _cache['optimum'] = optimum
    _cache['all']     = solve_all(w, v, W)
    step_slider.max   = len(w)  # n items => n+1 states (0..n)
    if step_slider.value > step_slider.max:
        step_slider.value = 0

def _update(*_):
    if 'all' not in _cache:
        _recompute()
    strat  = strategy_dd.value
    steps  = _cache['all'][strat]
    is_frac = (strat == 'Fractional (exact)')
    idx    = min(step_slider.value, len(steps) - 1)
    step   = steps[idx]

    with out:
        clear_output(wait=True)
        if show_summary_btn.value:
            fig, ax = plt.subplots(figsize=(8, 4))
            draw_summary(ax, _cache['all'], optimum=_cache.get('optimum'))
            plt.tight_layout()
            plt.show()
        else:
            fig, ax = plt.subplots(figsize=(9, 4.5))
            draw_step(ax, _cache['weights'], _cache['values'], _cache['W'],
                      step, is_fractional=is_frac)
            consideration_order = step['order']
            order_str = ' -> '.join(f'Item {i}' for i in consideration_order)
            ax.set_xlabel(f'Consideration order: {order_str}', fontsize=8,
                          color='#666')
            plt.tight_layout()
            plt.show()

def _on_instance_change(*_):
    _recompute()
    _update()

# Wire observers
strategy_dd.observe(_update, names='value')
step_slider.observe(_update, names='value')
show_summary_btn.observe(_update, names='value')
for w in (weights_txt, values_txt, cap_txt, opt_txt):
    w.observe(_on_instance_change, names='value')

# ── Layout ─────────────────────────────────────────────────────────────
input_box = widgets.VBox([
    widgets.HTML('<b>Custom instance</b> (comma-separated)'),
    widgets.HBox([weights_txt, values_txt]),
    widgets.HBox([cap_txt, opt_txt]),
])

controls = widgets.HBox([strategy_dd, step_slider, show_summary_btn])

ui = widgets.VBox([input_box, controls, out],
                  layout=widgets.Layout(gap='8px'))

_recompute()
_update()
display(ui)